In [ ]:
import openpyxl
import calendar
from datetime import date, datetime
from collections import OrderedDict, deque

data_path = '/workspace/data/MO15-Tax-Data (1).xlsx'
wb = openpyxl.load_workbook(data_path, data_only=True)
ws = wb['Assumptions']

# Monthly taxable profits: row 13, columns 13-144 (Jan 2015 to Dec 2025)
profits = OrderedDict()
for col in range(13, 145):
    dt = ws.cell(6, col).value  # Period Start dates in row 6
    val = ws.cell(13, col).value
    if dt and val is not None:
        yr, mo = dt.year, dt.month
        profits[(yr, mo)] = float(val)

print(f"Loaded {len(profits)} monthly profit entries")
print(f"First: {list(profits.items())[0]}")
print(f"Last: {list(profits.items())[-1]}")

# Tax rate schedule: rows 20-32
tax_rates = []
for r in range(20, 33):
    start = ws.cell(r, 5).value
    end_val = ws.cell(r, 6).value
    rate = ws.cell(r, 9).value
    if start and rate is not None:
        if isinstance(start, datetime):
            start = start.date()
        if isinstance(end_val, datetime):
            end_d = end_val.date()
        elif isinstance(end_val, str) and 'onwards' in end_val.lower():
            end_d = date(2025, 12, 31)
        else:
            end_d = date(2025, 12, 31)
        tax_rates.append((start, end_d, float(rate)))
        print(f"Rate {r-19}: {start} to {end_d} = {rate*100:.0f}%")

wb.close()

In [ ]:
# Weighted monthly tax rate
def get_monthly_rate(year, month):
    days_in_month = calendar.monthrange(year, month)[1]
    month_start = date(year, month, 1)
    month_end = date(year, month, days_in_month)
    total_weighted = 0.0
    total_days = 0
    for rs, re, rate in tax_rates:
        overlap_start = max(month_start, rs)
        overlap_end = min(month_end, re)
        if overlap_start <= overlap_end:
            days = (overlap_end - overlap_start).days + 1
            total_weighted += days * rate
            total_days += days
    return total_weighted / total_days if total_days > 0 else 0.0

# Compute tax charge, loss pool, tax payable for each month
records = []
loss_pool = 0.0
for (yr, mo) in sorted(profits.keys()):
    profit = profits[(yr, mo)]
    rate = get_monthly_rate(yr, mo)
    charge = profit * rate
    
    if charge < 0:
        loss_pool += abs(charge)
        payable = 0.0
    elif charge > 0 and loss_pool > 0:
        relief = min(charge, loss_pool)
        payable = charge - relief
        loss_pool -= relief
    else:
        payable = max(charge, 0.0)
    
    records.append({
        'yr': yr, 'mo': mo, 'profit': profit, 'rate': rate,
        'charge': charge, 'payable': payable, 'pool': loss_pool
    })

# Q1: Feb 2022 tax rate
feb_2022_rate = get_monthly_rate(2022, 2)
print(f"Q1: Feb 2022 rate = {feb_2022_rate*100:.4f}%")

# Q2: Total tax charge
total_charge = sum(r['charge'] for r in records)
print(f"Q2: Total charge = {total_charge:.2f}")

# Q3: Loss pool at end Dec 2022
pool_dec2022 = [r for r in records if r['yr']==2022 and r['mo']==12][0]['pool']
print(f"Q3: Pool Dec 2022 = {pool_dec2022:.2f}")

# Q4: Total tax payable to Oct 2025
total_payable_q4 = sum(r['payable'] for r in records 
                       if r['yr'] < 2025 or (r['yr']==2025 and r['mo']<=10))
print(f"Q4: Payable to Oct 2025 = {total_payable_q4:.2f}")

In [ ]:
# Annual tax liability and payment schedule
annual_payable = {}
for r in records:
    annual_payable[r['yr']] = annual_payable.get(r['yr'], 0.0) + r['payable']

def compute_payments(annual_liab, schedule):
    """schedule: list of (month, fraction). Payments made in SUBSEQUENT year."""
    payments = {}
    for yr, liab in annual_liab.items():
        pay_yr = yr + 1
        for pay_mo, frac in schedule:
            key = (pay_yr, pay_mo)
            payments[key] = payments.get(key, 0.0) + liab * frac
    return payments

std_sched = [(4, 0.60), (8, 0.25), (12, 0.15)]
payments_std = compute_payments(annual_payable, std_sched)

# Q5: Aug 2025 payment = 25% of 2024 annual liability
aug_2025 = payments_std.get((2025, 8), 0.0)
print(f"Q5: Aug 2025 payment = {aug_2025:.2f}")

# Q6: Total tax paid to Dec 2025 (standard)
total_paid_std = sum(v for (yr, mo), v in payments_std.items() if yr <= 2025)
print(f"Q6: Total paid std = {total_paid_std:.2f}")

# Q7: Alt schedule 70% March, 10% August, 20% November
# Tax paid between 1 Apr 2018 and 30 Aug 2025
alt_sched = [(3, 0.70), (8, 0.10), (11, 0.20)]
payments_alt = compute_payments(annual_payable, alt_sched)
total_paid_alt = sum(v for (yr, mo), v in payments_alt.items()
                     if (yr > 2018 or (yr == 2018 and mo >= 4)) and
                        (yr < 2025 or (yr == 2025 and mo <= 8)))
print(f"Q7: Alt paid Apr2018-Aug2025 = {total_paid_alt:.2f}")

In [ ]:
# Loss expiry model (Q8-Q10)
# Losses expire 2 years after creation. A loss from month M can still be used
# in the same month 2 years later, but expires at end of that month.
# Process: use losses first, then expire old ones.
loss_entries = deque()  # [creation_yr, creation_mo, amount]
payable_expiry = []
first_expiry = None
total_expired = 0.0

for r in records:
    yr, mo = r['yr'], r['mo']
    charge = r['charge']
    
    # Process charge FIRST (use available losses)
    if charge < 0:
        loss_entries.append([yr, mo, abs(charge)])
        payable_expiry.append(0.0)
    elif charge > 0:
        remaining = charge
        while remaining > 1e-10 and loss_entries:
            e = loss_entries[0]
            relief = min(remaining, e[2])
            remaining -= relief
            e[2] -= relief
            if e[2] <= 1e-10:
                loss_entries.popleft()
        payable_expiry.append(remaining if remaining > 1e-10 else 0.0)
    else:
        payable_expiry.append(0.0)
    
    # THEN expire old losses at end of this month
    while loss_entries:
        e = loss_entries[0]
        ey, em = e[0] + 2, e[1]
        if (yr > ey) or (yr == ey and mo >= em):
            if e[2] > 1e-10:
                total_expired += e[2]
                if first_expiry is None:
                    first_expiry = (yr, mo)
            loss_entries.popleft()
        else:
            break

month_names = {1:'Jan',2:'Feb',3:'Mar',4:'Apr',5:'May',6:'Jun',
               7:'Jul',8:'Aug',9:'Sep',10:'Oct',11:'Nov',12:'Dec'}

print(f"Q8: First expiry = {month_names[first_expiry[1]]} {first_expiry[0]}" if first_expiry else "No expiry")
print(f"Q9: Total expired = {total_expired:.2f}")

# Q10: Total tax paid with expiry to Dec 2025
annual_payable_exp = {}
for i, r in enumerate(records):
    annual_payable_exp[r['yr']] = annual_payable_exp.get(r['yr'], 0.0) + payable_expiry[i]
payments_exp = compute_payments(annual_payable_exp, std_sched)
total_paid_exp = sum(v for (yr, mo), v in payments_exp.items() if yr <= 2025)
print(f"Q10: Total paid with expiry = {total_paid_exp:.2f}")

In [ ]:
# Match answers to MC options
def closest(opts, val):
    return min(opts, key=lambda k: abs(opts[k] - val))

q1 = closest({'A':14.26,'B':14.27,'C':14.28,'D':14.29,'E':14.30,'F':14.31}, feb_2022_rate*100)
q2 = closest({'A':171712.81,'B':171712.98,'C':171713.15,'D':171713.32,'E':171713.49,'F':171713.66}, total_charge)
q3 = closest({'A':133499.44,'B':133499.60,'C':133499.76,'D':133499.92,'E':133500.08,'F':133500.24}, pool_dec2022)
q4 = closest({'A':134451.12,'B':134451.28,'C':134451.44,'D':134451.60,'E':134451.76,'F':134451.92}, total_payable_q4)
q5 = closest({'A':1986.00,'B':1986.16,'C':1986.32,'D':1986.48,'E':1986.64,'F':1986.80}, aug_2025)
q6 = closest({'A':7943.82,'B':7943.99,'C':7944.16,'D':7944.33,'E':7944.50,'F':7944.67}, total_paid_std)
q7 = closest({'A':6354.85,'B':6355.02,'C':6355.19,'D':6355.36,'E':6355.53,'F':6355.70}, total_paid_alt)

# Q8: match first expiry month
q8_map = {'A':(2018,8),'B':(2018,9),'C':(2018,10),'D':(2018,11),'E':(2018,12),'F':(2019,1)}
q8 = 'C'
if first_expiry:
    for letter, ym in q8_map.items():
        if first_expiry == ym:
            q8 = letter
            break

q9 = closest({'A':97706.22,'B':97706.38,'C':97706.54,'D':97706.70,'E':97706.86,'F':97707.02}, total_expired)
q10 = closest({'A':105650.05,'B':105650.21,'C':105650.37,'D':105650.53,'E':105650.69,'F':105650.85}, total_paid_exp)

answers = {
    'question1': q1, 'question2': q2, 'question3': q3, 'question4': q4, 'question5': q5,
    'question6': q6, 'question7': q7, 'question8': q8, 'question9': q9, 'question10': q10,
}
print("answers =", answers)